# 11-1절 연습 문제 풀이

이 노트북은 11-1절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch11/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

# 11장 공통 - Fashion-MNIST VAE / DCGAN 도우미
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
DATA_ROOT = '../../download'
LATENT_DIM = 16

def fashion_loader(batch_size=128, normalize_tanh=False):
    tf = [transforms.ToTensor()]
    if normalize_tanh: tf.append(transforms.Normalize((0.5,), (0.5,)))
    ds = datasets.FashionMNIST(root=DATA_ROOT, train=True, download=True,
                               transform=transforms.Compose(tf))
    return DataLoader(ds, batch_size=batch_size, shuffle=True)

class FashionVAE(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, 2, 1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2, 1), nn.ReLU())
        self.flatten_dim = 64 * 7 * 7
        self.fc_mu = nn.Linear(self.flatten_dim, latent_dim)
        self.fc_log_var = nn.Linear(self.flatten_dim, latent_dim)
        self.fc_z_to_fmap = nn.Linear(latent_dim, self.flatten_dim)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 3, 2, 1, output_padding=1), nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 3, 2, 1, output_padding=1), nn.Sigmoid())
    def encode(self, x):
        h = self.encoder(x).flatten(1)
        return self.fc_mu(h), self.fc_log_var(h)
    def reparameterize(self, mu, log_var):
        if self.training:
            std = torch.exp(0.5 * log_var)
            return mu + torch.randn_like(std) * std
        return mu
    def decode(self, z):
        return self.decoder(self.fc_z_to_fmap(z).view(-1, 64, 7, 7))
    def forward(self, x):
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        return self.decode(z), mu, log_var

bce_loss = nn.BCELoss(reduction='sum')
def vae_loss(recon, x, mu, log_var):
    bce = bce_loss(recon, x) / x.size(0)
    kl = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp()) / x.size(0)
    return bce + kl, bce, kl

def train_vae(model, epochs=10, lr=1e-3, latent_dim=16):
    loader = fashion_loader()
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for e in range(1, epochs + 1):
        model.train(); tot = n = 0
        for x, _ in loader:
            x = x.to(device)
            recon, mu, log_var = model(x)
            loss, bce, kl = vae_loss(recon, x, mu, log_var)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(x); n += len(x)
        if e % 5 == 0 or e == 1: print(f'  {e}/{epochs} 손실 {tot / n:.2f}')
    return model

## 연습 11-1

LATENT_DIM을 4, 16, 64로 각각 설정해 학습한 후 검증 데이터의 이미지 복원 품질과 표준 정규 분포에서 샘플링한 생성 이미지의 품질을 비교해 보자. 어떤 값이 가장 좋은 균형을 보이는지 정리해 보자.

In [ ]:
test_x = torch.stack([datasets.FashionMNIST(root=DATA_ROOT, train=False,
    download=True, transform=transforms.ToTensor())[i][0] for i in range(8)]).to(device)
for latent in (4, 16, 64):
    torch.manual_seed(SEED)
    print(f'[LATENT_DIM={latent}]')
    m = train_vae(FashionVAE(latent), epochs=10)
    m.eval()
    with torch.no_grad():
        recon, _, _ = m(test_x)
        gen = m.decode(torch.randn(8, latent, device=device))
    viz.plot_images(list(test_x.cpu()) + list(recon.cpu()) + list(gen.cpu()),
                    ['원본'] * 8 + [f'복원({latent})'] * 8 + ['생성'] * 8,
                    images_per_row=8)

잠재 차원이 커질수록 **복원 품질은 좋아지지만 생성 품질은 반드시 그렇지 않다**. 차원이 커지면 KL 발산이 잠재 공간 전체를 표준 정규 분포로 정리하기 어려워져, 무작위로 뽑은 벡터가 데이터가 없는 영역에 떨어질 확률이 높아진다.

Fashion-MNIST에서는 16 안팎이 복원과 생성의 균형이 좋다.

## 연습 11-2

Fashion-MNIST 데이터셋을 분류하는 모델을 만든 후, [코드 11-6]으로 생성한 이미지([그림 11-4])의 분류 결과를 확인해 보자.

In [ ]:
# 분류기를 학습한 뒤 VAE 생성 이미지를 분류해 본다.
CLASSES = ['티셔츠/탑','바지','풀오버','드레스','코트','샌들','셔츠','스니커즈','가방','앵클부츠']
torch.manual_seed(SEED)
clf = nn.Sequential(nn.Conv2d(1, 32, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),
                    nn.Conv2d(32, 64, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),
                    nn.Flatten(), nn.Linear(64 * 7 * 7, 10)).to(device)
opt = torch.optim.Adam(clf.parameters(), lr=1e-3); crit = nn.CrossEntropyLoss()
loader = fashion_loader()
for e in range(3):
    clf.train()
    for x, y in loader:
        loss = crit(clf(x.to(device)), y.to(device))
        opt.zero_grad(); loss.backward(); opt.step()

torch.manual_seed(SEED)
vae = train_vae(FashionVAE(16), epochs=10)
vae.eval(); clf.eval()
with torch.no_grad():
    gen = vae.decode(torch.randn(16, 16, device=device))
    probs = torch.softmax(clf(gen), dim=1)
pred = probs.argmax(1)
viz.plot_images(list(gen.cpu()),
                [f'{CLASSES[p]}\n{probs[i, p]*100:.0f}%' for i, p in enumerate(pred)],
                images_per_row=8)
import collections
print(collections.Counter(CLASSES[p] for p in pred.tolist()))

생성 이미지의 분류 결과를 보면 **어떤 클래스가 잘 생성되는지** 알 수 있다. 보통 형태가 단순한 바지·가방·티셔츠는 높은 확률로 분류되고, 셔츠·코트·풀오버처럼 서로 닮은 클래스는 확률이 낮게 흩어진다.

이렇게 별도 분류기로 생성 품질을 재는 방식이 11장 학습 노트에서 소개한 인셉션 점수·FID의 기본 아이디어다.

## 연습 11-3

FashionVAE 모델을 합성곱 기반 인코더와 디코더 대신 7장처럼 다층 퍼셉트론을 사용하도록 수정하고, 복원 품질과 생성 이미지 품질을 비교해 보자. 합성곱 신경망이 어떤 측면에서 우위를 보이는지 정리해 보자.

In [ ]:
class MLPVae(nn.Module):
    def __init__(self, latent_dim=16, hidden=256):
        super().__init__()
        self.enc = nn.Sequential(nn.Flatten(), nn.Linear(784, hidden), nn.ReLU())
        self.fc_mu = nn.Linear(hidden, latent_dim)
        self.fc_log_var = nn.Linear(hidden, latent_dim)
        self.dec = nn.Sequential(nn.Linear(latent_dim, hidden), nn.ReLU(),
                                 nn.Linear(hidden, 784), nn.Sigmoid())
        self.latent_dim = latent_dim
    def encode(self, x):
        h = self.enc(x); return self.fc_mu(h), self.fc_log_var(h)
    def reparameterize(self, mu, log_var):
        if self.training:
            return mu + torch.randn_like(mu) * torch.exp(0.5 * log_var)
        return mu
    def decode(self, z): return self.dec(z).view(-1, 1, 28, 28)
    def forward(self, x):
        mu, lv = self.encode(x); return self.decode(self.reparameterize(mu, lv)), mu, lv

for name, m in [('합성곱 VAE', FashionVAE(16)), ('다층 퍼셉트론 VAE', MLPVae(16))]:
    torch.manual_seed(SEED)
    print(f'[{name}] 파라미터 {sum(p.numel() for p in m.parameters()):,}개')
    trained = train_vae(m, epochs=10)
    trained.eval()
    with torch.no_grad():
        recon, _, _ = trained(test_x)
        gen = trained.decode(torch.randn(8, 16, device=device))
    viz.plot_images(list(recon.cpu()) + list(gen.cpu()),
                    [f'{name[:2]} 복원'] * 8 + ['생성'] * 8, images_per_row=8)

합성곱 VAE가 우위를 보이는 지점은 두 가지다.

1. **공간 구조 활용**: 이웃 픽셀의 관계를 필터로 학습하므로 옷의 윤곽이 또렷하다. 다층 퍼셉트론은 픽셀을 일렬로 펼쳐 위치 관계를 처음부터 배워야 한다.
2. **파라미터 효율**: 필터를 이미지 전체에 공유하므로 적은 파라미터로 더 좋은 품질을 낸다.

다층 퍼셉트론 VAE는 전반적으로 흐릿하고 세부가 뭉개진다.

## 연습 11-4

[도전 문제] 인터넷을 검색하면 CelebA 등의 실습에 활용할 수 있는 여러 얼굴 데이터셋을 찾을 수 있다. 그중 적절한 컬러 데이터셋을 하나 선택해 VAE 모델을 만들고 학습한 후, 임의의 두 사람 얼굴 이미지 사이를 잠재 공간에서 보간해, 두 얼굴의 중간 형태에 해당하는 이미지를 만들어 보자. CPU 환경에서 이 모델을 학습하는 데 꽤 오랜 시간이 걸릴 수 있으므로 구글 코랩 등 하드웨어 가속기 장치를 사용할 수 있는 환경을 권장한다.

### 풀이

컬러 얼굴 데이터셋(CelebA 등)으로 VAE를 학습하려면 다음을 바꿔야 한다.

1. **입력 채널 1 → 3**, 이미지 크기를 64×64로 맞춘다(`transforms.Resize`, `CenterCrop`).
2. **합성곱 단계를 늘려** 64 → 32 → 16 → 8로 줄이고, 디코더는 역순으로 키운다.
3. **잠재 차원을 128 안팎**으로 키운다. 얼굴은 의류보다 정보량이 많다.
4. **복원 손실**: 픽셀이 0~1이면 BCE를 그대로 써도 되고, MSE가 더 자연스러운 경우도 많다.

학습 후 두 얼굴의 `mu`를 선형 보간하면 한 얼굴이 다른 얼굴로 부드럽게 변하는 결과를 얻는다.

> **주의**: CelebA는 `torchvision`의 자동 내려받기가 구글 드라이브 할당량 문제로 실패하는 경우가 잦다. 허깅페이스 미러나 캐글에서 직접 받아 `data/`에 두고 쓰는 편이 안전하다.